In [1]:
import os
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Rutas del dataset
DATASET_DIR = "./dataset_animals10" # Ruta del dataset
OUTPUT_DIR = "./processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mapeo de subcarpetas a etiquetas numéricas
# (Animals-10 suele venir con carpetas en italiano: cane, gatto, etc.)
CLASSES = sorted(os.listdir(DATASET_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

valid_paths = []
labels = []
corrupted_files = []

print("--- Paso 1: Limpieza e Inspección (EDA) ---")
for class_name in CLASSES:
    class_dir = os.path.join(DATASET_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    
    for fname in os.listdir(class_dir):
        fpath = os.path.join(class_dir, fname)
        
        # Verificar que sea una imagen válida usando OpenCV
        try:
            img = cv2.imread(fpath)
            if img is None:
                corrupted_files.append(fpath)
            else:
                valid_paths.append(fpath)
                labels.append(class_to_idx[class_name])
        except Exception as e:
            corrupted_files.append(fpath)

print(f"Imágenes válidas encontradas: {len(valid_paths)}")
print(f"Archivos dañados o ilegibles omitidos: {len(corrupted_files)}")

# 2. División del Dataset (70% Train, 15% Val, 15% Test)
print("\n--- Paso 2: División de Datos (70 / 15 / 15) ---")
# Primera división: 70% Train, 30% Temporal (Val + Test)
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    valid_paths, 
    labels, 
    test_size=0.30, 
    random_state=42, 
    stratify=labels # Mantiene la proporción de cada especie en los 3 splits
)

# Segunda división del 30% restante: 50% Val y 50% Test (equivale al 15% del total cada uno)
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, 
    y_temp, 
    test_size=0.50, 
    random_state=42, 
    stratify=y_temp
)

print(f"Conjunto de Entrenamiento (Train): {len(X_train_paths)} imágenes")
print(f"Conjunto de Validación (Val):       {len(X_val_paths)} imágenes")
print(f"Conjunto de Prueba (Test):          {len(X_test_paths)} imágenes")

# 3. Entrega de Listas / Exportar Archivos
print("\n--- Paso 3: Guardar listas de datos ---")

# Creamos DataFrames para guardar las rutas y etiquetas
df_train = pd.DataFrame({'path': X_train_paths, 'label': y_train})
df_val   = pd.DataFrame({'path': X_val_paths,   'label': y_val})
df_test  = pd.DataFrame({'path': X_test_paths,  'label': y_test})

# Guardar en CSV para compartir fácilmente entre los integrantes
df_train.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
df_test.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)

print("¡Listo! Los archivos 'train_split.csv', 'val_split.csv' y 'test_split.csv' han sido guardados.")

--- Paso 1: Limpieza e Inspección (EDA) ---
Imágenes válidas encontradas: 26179
Archivos dañados o ilegibles omitidos: 0

--- Paso 2: División de Datos (70 / 15 / 15) ---
Conjunto de Entrenamiento (Train): 18325 imágenes
Conjunto de Validación (Val):       3927 imágenes
Conjunto de Prueba (Test):          3927 imágenes

--- Paso 3: Guardar listas de datos ---
¡Listo! Los archivos 'train_split.csv', 'val_split.csv' y 'test_split.csv' han sido guardados.


In [2]:
import cv2
import numpy as np
from skimage.feature import local_binary_pattern, hog

def extract_color_histogram(image, bins=(8, 8, 8)):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

def extract_lbp_features(image_gray, radius=1, n_points=8):
    lbp = local_binary_pattern(image_gray, n_points, radius, method="uniform")
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    return hist

def extract_hog_features(image_gray):
    resized = cv2.resize(image_gray, (64, 64))
    features = hog(
        resized,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )
    return features

def extract_all_features(image_path_or_matrix):
    if isinstance(image_path_or_matrix, str):
        image = cv2.imread(image_path_or_matrix)
    else:
        image = image_path_or_matrix

    image = cv2.resize(image, (128, 128))
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    color_hist = extract_color_histogram(image)
    lbp_hist = extract_lbp_features(image_gray)
    hog_feat = extract_hog_features(image_gray)

    return np.hstack([color_hist, lbp_hist, hog_feat])

In [3]:
import pandas as pd
import numpy as np

# Cargar las listas que preparó Jasciel
df_train = pd.read_csv("./processed_data/train_split.csv")

# Extraer las características únicamente del conjunto de entrenamiento
X_train_features = []
y_train = df_train['label'].values

for img_path in df_train['path']:
    # Usas tu función de extracción previa
    feat = extract_all_features(img_path)
    X_train_features.append(feat)

X_train_features = np.array(X_train_features)

In [4]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 1. Definir los modelos base y sus mallas de hiperparámetros (GridSearch)
param_grids = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.1, 1.0, 10.0]
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'max_depth': [10, 20, None]
        }
    },
    'SVM': {
        'model': SVC(random_state=42),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear']
        }
    }
}

best_models = {}

print("--- Entrenando Modelos Clásicos con GridSearchCV ---")
for model_name, config in param_grids.items():
    print(f"\nEntrenando {model_name}...")
    grid = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=3,                 # Validación cruzada de 3 pliegues
        scoring='accuracy',
        n_jobs=-1,            # Usa todos los núcleos de tu procesador
        verbose=1
    )
    grid.fit(X_train_features, y_train)
    
    best_models[model_name] = grid.best_estimator_
    print(f"Mejores hiperparámetros para {model_name}: {grid.best_params_}")
    print(f"Mejor Accuracy en CV: {grid.best_score_:.4f}")

# Guardar los modelos entrenados para que Betzie u otros puedan cargarlos
os.makedirs("./saved_models", exist_ok=True)
for name, model in best_models.items():
    joblib.dump(model, f"./saved_models/{name}_best.pkl")

print("\n¡Modelos guardados exitosamente en ./saved_models!")

--- Entrenando Modelos Clásicos con GridSearchCV ---

Entrenando LogisticRegression...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Mejores hiperparámetros para LogisticRegression: {'C': 0.1}
Mejor Accuracy en CV: 0.5012

Entrenando RandomForest...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Mejores hiperparámetros para RandomForest: {'max_depth': 20, 'n_estimators': 100}
Mejor Accuracy en CV: 0.5044

Entrenando SVM...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Mejores hiperparámetros para SVM: {'C': 10.0, 'kernel': 'rbf'}
Mejor Accuracy en CV: 0.5566

¡Modelos guardados exitosamente en ./saved_models!


In [5]:
# 1. Cargar conjunto de prueba (Test Split)
df_test = pd.read_csv("./processed_data/test_split.csv")
y_test = df_test['label'].values

# 2. Extraer características del conjunto de prueba
X_test_features = []
for img_path in df_test['path']:
    feat = extract_all_features(img_path)
    X_test_features.append(feat)

X_test_features = np.array(X_test_features)

# 3. Generar predicciones con los modelos entrenados
predictions = {}
for model_name, model in best_models.items():
    y_pred = model.predict(X_test_features)
    predictions[model_name] = y_pred
    print(f"Predicciones listas para {model_name}.")

# Guardar X_test, y_test y las predicciones para las métricas de Betzie
np.savez(
    "./processed_data/test_features_and_preds.npz",
    X_test=X_test_features,
    y_test=y_test,
    **predictions
)
print("¡Archivo de evaluación 'test_features_and_preds.npz' guardado para Betzie!")

Predicciones listas para LogisticRegression.
Predicciones listas para RandomForest.
Predicciones listas para SVM.
¡Archivo de evaluación 'test_features_and_preds.npz' guardado para Betzie!
